`# R&E 연구 검증 방법론 3편: Three+ Features (다변량 분석)

변수가 **3개 이상**일 때의 분석. 현실 데이터는 거의 항상 다변량이다.

"여러 변수가 동시에 결과에 어떻게 영향을 미치는가?"
"고차원 데이터에서 숨겨진 구조를 발견할 수 있는가?"

```
3+ Features
|
|-- [A] 관계 분석
|     |-- 다중 회귀 (Multiple Regression)
|     |-- 다중공선성 진단 (VIF)
|     |-- 편상관 (Partial Correlation)
|
|-- [B] 차원 축소 & 시각화
|     |-- PCA (주성분 분석)
|     |-- t-SNE / UMAP
|
|-- [C] 군집화 (비지도 학습)
|     |-- K-Means
|     |-- DBSCAN
|     |-- 계층적 군집화 (Dendrogram)
|
|-- [D] 분류 & 예측 (지도 학습)
|     |-- Logistic Regression
|     |-- Random Forest
|     |-- 교차 검증 (Cross-Validation)
|
|-- [E] 모델 해석
      |-- Feature Importance
      |-- Permutation Importance
      |-- SHAP
```


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = sns.load_dataset('penguins').dropna()
numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
print(f'데이터 수: {len(df)}, 수치형 변수: {len(numeric_cols)}개')
df.head()

데이터 수: 333, 수치형 변수: 4개


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male


---
## 3-A. 다중 회귀 분석 (Multiple Regression)

독립변수가 2개 이상일 때의 회귀.
**"여러 변수 중 어떤 것이 종속변수에 영향을 미치는가?"**

2편의 단순 회귀와의 차이:
| | 단순 회귀 | 다중 회귀 |
|--|---------|---------|
| 독립변수 | 1개 | 2개+ |
| 모델 | y = ax + b | y = a1*x1 + a2*x2 + ... + b |
| 주의점 | - | **다중공선성** 확인 필수 |

### 다중공선성 (Multicollinearity)
독립변수들끼리 높은 상관이 있으면 회귀 계수가 불안정해진다.
**VIF (Variance Inflation Factor)**로 진단:
- VIF < 5: 문제 없음
- VIF 5~10: 주의
- VIF > 10: 심각한 공선성 -> 변수 제거 또는 PCA

### Agent 지시 예시
> "bill_length, bill_depth, flipper_length로 body_mass를 예측하는 다중 회귀를 해줘.
> 각 변수의 회귀 계수, p-value, R-squared를 보고하고 VIF도 확인해줘."


In [ ]:
# 다중 회귀: statsmodels (통계적 상세 결과)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']]
y = df['body_mass_g']

# 상수항 추가 (절편)
X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()
print(model.summary())

In [ ]:
# VIF (다중공선성 진단)
print('=== VIF (Variance Inflation Factor) ===')
print()
for i, col in enumerate(X.columns):
    vif = variance_inflation_factor(X.values, i)
    status = 'OK' if vif < 5 else ('주의' if vif < 10 else '심각!')
    print(f'  {col:20s}: VIF = {vif:.2f}  [{status}]')
print()
print('  VIF < 5: 문제 없음')
print('  VIF 5~10: 주의')
print('  VIF > 10: 변수 제거 또는 PCA 고려')

In [ ]:
# 편상관 (Partial Correlation)
# "다른 변수들의 영향을 제거한 후의 순수한 관계"
# 예: flipper와 body_mass의 상관에서 bill_length, bill_depth의 영향을 제거

def partial_corr(df, x, y, covariates):
    """x와 y의 편상관: covariates의 영향을 제거한 순수 상관"""
    from sklearn.linear_model import LinearRegression
    # x에서 covariates의 영향 제거 -> 잔차
    cov = df[covariates].values
    model_x = LinearRegression().fit(cov, df[x])
    model_y = LinearRegression().fit(cov, df[y])
    resid_x = df[x] - model_x.predict(cov)
    resid_y = df[y] - model_y.predict(cov)
    return stats.pearsonr(resid_x, resid_y)

print('=== 편상관 분석 ===')
print('flipper_length와 body_mass의 관계 (다른 변수 통제)')
print()

# 단순 상관
r_simple, p_simple = stats.pearsonr(df['flipper_length_mm'], df['body_mass_g'])
print(f'  단순 상관:   r = {r_simple:.3f}, p = {p_simple:.2e}')

# 편상관 (bill_length, bill_depth 통제)
r_partial, p_partial = partial_corr(df, 'flipper_length_mm', 'body_mass_g',
                                     ['bill_length_mm', 'bill_depth_mm'])
print(f'  편상관:      r = {r_partial:.3f}, p = {p_partial:.2e}')
print()
print('  -> 다른 변수를 통제하면 순수한 관계가 얼마나 변하는지 확인')
print('     제3변수의 영향을 배제한 "진짜 관계"를 볼 수 있다')

---
## 3-B. 차원 축소 & 시각화

고차원(변수가 많은) 데이터를 2~3차원으로 압축해서 **눈으로 보는** 방법.

| 방법 | 특징 | 용도 |
|------|------|------|
| **PCA** | 분산을 최대한 보존하며 축소. 선형 | 변수 요약, 노이즈 제거, 공선성 해결 |
| **t-SNE** | 가까운 점은 가깝게, 먼 점은 멀게. 비선형 | 군집 시각화, 패턴 발견 |
| **UMAP** | t-SNE보다 빠르고 전역 구조 보존 | 대규모 데이터 시각화 |

### PCA 핵심 개념
- 원래 변수들의 **선형 조합**으로 새로운 축(주성분)을 만든다
- PC1이 가장 많은 분산을 설명, PC2가 그 다음, ...
- **설명 분산 비율**: PC 몇 개로 원본의 몇 %를 설명하는가?
- **적재량(loadings)**: 각 원래 변수가 주성분에 기여하는 정도

### Agent 지시 예시
> "수치형 변수들에 PCA를 적용해줘.
> 설명 분산 비율, 스크리 플롯, PC1-PC2 산점도(종별 색구분)를 보여줘.
> 각 PC에 어떤 원래 변수가 기여하는지 적재량도 알려줘."


In [ ]:
# PCA
from sklearn.decomposition import PCA

# 표준화 (PCA 전에 필수 - 단위가 다르면 큰 값이 지배)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numeric_cols])

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# 설명 분산 비율
print('=== PCA 설명 분산 비율 ===')
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_,
                                    np.cumsum(pca.explained_variance_ratio_))):
    print(f'  PC{i+1}: {var:.3f} ({var*100:.1f}%)  누적: {cum:.3f} ({cum*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 스크리 플롯
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1),
            pca.explained_variance_ratio_, edgecolor='black', alpha=0.7)
axes[0].plot(range(1, len(pca.explained_variance_ratio_)+1),
             np.cumsum(pca.explained_variance_ratio_), 'ro-')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].set_xticks(range(1, len(numeric_cols)+1))

# PC1-PC2 산점도 (종별 색구분)
species_list = df['species'].unique()
colors = ['blue', 'orange', 'green']
for sp, c in zip(species_list, colors):
    mask = df['species'] == sp
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=c, label=sp, alpha=0.6)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA: PC1 vs PC2')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# PCA 적재량 (Loadings): 각 원래 변수의 기여도
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(numeric_cols))],
    index=numeric_cols
)
print('=== PCA Loadings (적재량) ===')
print(loadings.round(3))
print()
print('해석 예시:')
for i in range(2):
    dominant = loadings[f'PC{i+1}'].abs().idxmax()
    print(f'  PC{i+1}: {dominant}의 영향이 가장 큼 (loading={loadings.loc[dominant, f"PC{i+1}"]:.3f})')

In [ ]:
# t-SNE: 비선형 차원 축소 (군집 시각화에 강력)
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
for sp, c in zip(species_list, colors):
    mask = df['species'] == sp
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=c, label=sp, alpha=0.6)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('t-SNE Visualization')
plt.legend()
plt.show()

print('t-SNE 주의사항:')
print('  - 축의 값 자체는 의미 없음 (거리 비교 불가)')
print('  - perplexity 파라미터에 결과가 민감')
print('  - 전역 구조보다 지역 구조(군집)를 잘 보여줌')
print('  - 매번 결과가 다를 수 있음 (random_state 고정 필요)')

---
## 3-C. 군집화 (Clustering) - 비지도 학습

정답 레이블 없이 **데이터 자체의 구조**를 발견하는 방법.
"이 데이터에서 자연스러운 그룹이 있는가?"

| 방법 | 특징 | 장점 | 단점 |
|------|------|------|------|
| **K-Means** | K개 군집 중심으로 분할 | 빠르고 직관적 | K를 미리 지정해야 함 |
| **DBSCAN** | 밀도 기반 군집화 | 군집 수 자동, 이상치 탐지 | eps, min_samples 조정 필요 |
| **계층적 (Hierarchical)** | 트리 구조로 병합/분할 | 덴드로그램으로 시각화 | 대규모 데이터에 느림 |

### 군집 수(K) 결정: Elbow Method + Silhouette Score

### Agent 지시 예시
> "수치형 변수를 표준화하고 K-Means 군집화를 해줘.
> Elbow method와 Silhouette score로 최적 K를 찾고,
> 군집 결과를 PCA 2D로 시각화해줘."


In [ ]:
# K-Means: Elbow Method + Silhouette Score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias = []
sil_scores = []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('K (Number of Clusters)')
axes[0].set_ylabel('Inertia (Within-cluster SS)')
axes[0].set_title('Elbow Method')

# Silhouette
axes[1].plot(K_range, sil_scores, 'ro-')
axes[1].set_xlabel('K (Number of Clusters)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)')

plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f'Silhouette 최적 K = {best_k} (score = {max(sil_scores):.3f})')

In [ ]:
# K-Means 결과 시각화 (PCA 2D)
best_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = best_km.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-Means 군집
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.6)
axes[0].set_title(f'K-Means (K={best_k})')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# 실제 종과 비교
species_num = LabelEncoder().fit_transform(df['species'])
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=species_num, cmap='viridis', alpha=0.6)
axes[1].set_title('Actual Species')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1], label='Species')

plt.tight_layout()
plt.show()

# 군집과 실제 종의 일치도
ct = pd.crosstab(df['species'], cluster_labels, colnames=['Cluster'])
print('=== 군집 vs 실제 종 ===')
print(ct)

In [ ]:
# DBSCAN: 밀도 기반 군집화
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=1.2, min_samples=5)
db_labels = db.fit_predict(X_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels, cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Cluster (-1 = noise)')
plt.title(f'DBSCAN: {n_clusters} clusters, {n_noise} noise points')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print(f'군집 수: {n_clusters}')
print(f'노이즈(이상치): {n_noise}개')
print()
print('DBSCAN 장점: 군집 수를 자동 결정, 이상치를 -1로 분리')
print('DBSCAN 단점: eps와 min_samples 파라미터에 민감')

In [ ]:
# 계층적 군집화 (Hierarchical Clustering) + 덴드로그램
from scipy.cluster.hierarchy import dendrogram, linkage

# 표본 추출 (덴드로그램은 전체 표시하면 너무 복잡)
sample_idx = np.random.RandomState(42).choice(len(X_scaled), 50, replace=False)
X_sample = X_scaled[sample_idx]
species_sample = df['species'].values[sample_idx]

linkage_matrix = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(linkage_matrix,
           labels=species_sample,
           leaf_rotation=90,
           leaf_font_size=8)
plt.title('Hierarchical Clustering Dendrogram (50 samples)')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

print('덴드로그램 읽는 법:')
print('  - 아래에서 위로: 가장 가까운 점들이 먼저 병합')
print('  - 수평선 높이: 병합될 때의 거리 (높을수록 멀리 떨어진 군집)')
print('  - 특정 높이에서 수평으로 자르면 -> 그 수의 군집이 됨')

---
## 3-D. 분류 & 예측 (지도 학습)

정답 레이블이 있을 때, **새로운 데이터의 레이블을 예측**하는 방법.

| 방법 | 특징 | 장점 |
|------|------|------|
| **Logistic Regression** | 선형 경계, 확률 출력 | 해석 용이, 기준선(baseline) |
| **Random Forest** | 앙상블(여러 결정 트리 투표) | 비선형, Feature Importance 제공 |
| **SVM** | 마진 최대화 경계 | 고차원에 강함 |
| **KNN** | 가장 가까운 K개 이웃 투표 | 단순, 직관적 |

### 핵심: 교차 검증 (Cross-Validation)
- 데이터를 K등분하여 K번 학습/평가를 반복
- 과적합(overfitting) 방지, 성능의 안정적 추정
- **절대 테스트 데이터로 학습하지 않는다**

### Agent 지시 예시
> "수치형 변수로 종(species)을 분류하는 Random Forest를 만들어줘.
> 5-fold 교차 검증으로 정확도를 보고하고, Feature Importance도 그려줘."


In [ ]:
# Random Forest 분류 + 교차 검증
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

X = df[numeric_cols].values
y = LabelEncoder().fit_transform(df['species'])
species_names = df['species'].unique()

# 5-fold 교차 검증
rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')

print('=== Random Forest 5-Fold CV ===')
print(f'  Accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')
print(f'  각 Fold: {[f"{s:.3f}" for s in scores]}')
print()

# 교차 검증 예측으로 혼동 행렬
y_pred = cross_val_predict(rf, X, y, cv=5)
print('=== Classification Report ===')
print(classification_report(y, y_pred, target_names=species_names))

In [ ]:
# 혼동 행렬 시각화
cm = confusion_matrix(y, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=species_names, yticklabels=species_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (5-Fold CV)')
plt.tight_layout()
plt.show()

print('혼동 행렬 읽는 법:')
print('  - 대각선: 맞춘 개수')
print('  - 비대각선: 틀린 개수 (어떤 종을 어떤 종으로 오분류했는지)')

---
## 3-E. 모델 해석 (Feature Importance & SHAP)

모델이 **왜 그런 예측을 했는가?** 단순히 정확도만 보고하면 연구가 아니다.

| 방법 | 원리 | 장점 |
|------|------|------|
| **Feature Importance** (RF 내장) | 각 변수가 분할에 기여한 정도 | 빠름, 직관적 |
| **Permutation Importance** | 변수를 섞었을 때 성능 하락 정도 | 모델 무관, 더 신뢰성 |
| **SHAP** | 게임이론 기반 각 변수의 기여도 | 개별 예측 해석 가능, 가장 상세 |

### 주의: Feature Importance의 함정
- RF 내장 Importance는 **상관된 변수에 편향**될 수 있다
- 상관된 변수 A, B가 있으면 둘 다 중요도가 낮게 나옴 (서로 나눠가짐)
- **Permutation Importance**가 더 신뢰할 만하다

### Agent 지시 예시
> "Random Forest의 Feature Importance와 Permutation Importance를 비교해줘.
> SHAP summary plot도 그려줘."


In [ ]:
# Feature Importance 비교
from sklearn.inspection import permutation_importance

# 모델 학습 (전체 데이터로, 해석 목적)
rf.fit(X, y)

# 1) RF 내장 Feature Importance
fi_builtin = rf.feature_importances_

# 2) Permutation Importance
perm_result = permutation_importance(rf, X, y, n_repeats=30, random_state=42)
fi_perm = perm_result.importances_mean

# 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

idx = np.argsort(fi_builtin)
axes[0].barh(range(len(numeric_cols)), fi_builtin[idx], edgecolor='black', alpha=0.7)
axes[0].set_yticks(range(len(numeric_cols)))
axes[0].set_yticklabels([numeric_cols[i] for i in idx])
axes[0].set_title('RF Built-in Feature Importance')

idx2 = np.argsort(fi_perm)
axes[1].barh(range(len(numeric_cols)), fi_perm[idx2], edgecolor='black', alpha=0.7)
axes[1].set_yticks(range(len(numeric_cols)))
axes[1].set_yticklabels([numeric_cols[i] for i in idx2])
axes[1].set_title('Permutation Feature Importance')

plt.tight_layout()
plt.show()

print('=== Feature Importance 비교 ===')
for col, bi, pi in zip(numeric_cols, fi_builtin, fi_perm):
    print(f'  {col:20s}: Built-in={bi:.3f}, Permutation={pi:.3f}')

In [ ]:
# SHAP (SHapley Additive exPlanations)
# Colab: !pip install shap
try:
    import shap

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X)

    # Summary Plot: 전체 변수의 중요도 + 방향
    plt.figure()
    shap.summary_plot(shap_values, X, feature_names=numeric_cols,
                      class_names=list(species_names), show=False)
    plt.tight_layout()
    plt.show()

    print('SHAP Summary Plot 읽는 법:')
    print('  - x축: SHAP 값 (양수=해당 클래스 예측에 기여, 음수=반대)')
    print('  - 색상: 변수 값의 크기 (빨강=높은 값, 파랑=낮은 값)')
    print('  - 각 점은 하나의 데이터 포인트')

except ImportError:
    print('shap 패키지가 설치되지 않았습니다.')
    print('Colab에서: !pip install shap')
    print()
    print('SHAP이 제공하는 것:')
    print('  1) 각 변수가 각 예측에 얼마나 기여했는지 수치화')
    print('  2) Summary plot: 전체 변수 중요도 + 방향')
    print('  3) Force plot: 개별 예측의 이유 시각화')
    print('  4) Dependence plot: 특정 변수와 SHAP 값의 관계')

---
## 3-F. 실전 분석 워크플로우

실제 연구에서 다변량 분석을 수행하는 전체 흐름.

```
[1] 데이터 탐색 (EDA)
    |-- 각 변수 단변량 분석 (1편)
    |-- 주요 변수 쌍 이변량 분석 (2편)
    |-- 상관행렬로 전체 관계 조망
    |
[2] 전처리
    |-- 결측치 처리 (dropna / 대체)
    |-- 이상치 처리 (제거 / 변환)
    |-- 표준화 (StandardScaler) - 단위가 다른 변수들
    |-- 인코딩 (범주형 -> 숫자) - LabelEncoder / OneHotEncoder
    |
[3] 차원 축소 & 군집화 (탐색적)
    |-- PCA / t-SNE로 구조 시각화
    |-- K-Means / DBSCAN으로 자연적 그룹 발견
    |
[4] 모델링
    |-- 연속 종속변수: 다중 회귀 + VIF
    |-- 범주 종속변수: 분류 (RF, Logistic, SVM)
    |-- 교차 검증으로 성능 평가
    |
[5] 해석
    |-- Feature Importance / SHAP
    |-- 통계적 유의성 보고
    |-- 효과 크기 보고
    |
[6] 보고
    |-- 결과 재현 가능하게 정리 (random_state 고정)
    |-- 한계점 명시
    |-- 코드와 데이터 공유
```


---
## 3편 요약: 다변량 분석 선택 가이드

```
변수 3개 이상 -> 목적이 무엇인가?
|
|-- 관계/영향력 파악
|     |-- 종속변수 있음 (수치): 다중 회귀 + VIF
|     |-- 종속변수 있음 (범주): 분류 (RF, Logistic)
|     |-- 종속변수 없음: 편상관, 상관행렬
|
|-- 구조/패턴 발견
|     |-- 차원 축소: PCA (설명), t-SNE (시각화)
|     |-- 군집화: K-Means (K 알 때), DBSCAN (K 모를 때)
|     |-- 계층적: 덴드로그램
|
|-- 모델 해석
      |-- 어떤 변수가 중요?: Feature Importance, Permutation
      |-- 왜 이 예측?: SHAP
```

### Agent 통합 지시 예시
> "이 데이터를 종합적으로 분석해줘:
> 1) 상관행렬로 변수 간 관계를 보고
> 2) PCA로 차원 축소해서 시각화하고
> 3) [종속변수]를 예측하는 모델을 만들어서 교차 검증하고
> 4) Feature Importance와 SHAP으로 어떤 변수가 중요한지 해석해줘."

### 1~3편 전체 분석 요약

| 변수 수 | 핵심 질문 | 대표 방법 |
|---------|----------|----------|
| **1개** | "어떻게 생겼는가?" | 기술통계, 정규성, KDE, 이상치 |
| **2개** | "둘 사이에 관계가 있는가?" | 상관, 회귀, t-test, ANOVA, 카이제곱 |
| **3개+** | "여러 변수가 동시에 어떻게 작용하는가?" | 다중회귀, PCA, 군집화, 분류, SHAP |
